# Självstudieövningar – Vecka 7

De här övningarna är till för att du ska repetera och befästa det vi har gått igenom under veckan samt introducera några nya begrepp på ett tryggt sätt.

Det innebär att vissa av uppgifterna kan du behöva söka efter information själv hur du löser. Här kan du använda Google eller AI om du vill men dokumentationerna är din bästa vän när du lär dig.
- Python: https://docs.python.org/3.13/
- Matplotlib: https://matplotlib.org/stable/
- Pandas: https://pandas.pydata.org/
- NumPy: https://numpy.org/

Läs först igenom vad som ska göras i respektive uppgift och fyll sedan i kodblocket nedanför, eller gör uppgiften i en separat fil.

**Detta lämnas inte in för rättning.** Arbeta i din egen takt, testa dig fram, använd dokumentationen när du kör fast och ta med eventuella frågor till nästa lektionstillfälle.

## (Kör först) Skapa veckans dataset

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(123)

n = 800
cities = np.array(["Göteborg", "Malmö", "Stockholm", "Uppsala"])
genders = np.array(["F", "M"])
groups = np.array(["A", "B"])

city = rng.choice(cities, size=n, p=[0.28, 0.22, 0.40, 0.10])
gender = rng.choice(genders, size=n)
age = rng.integers(18, 65, size=n)
group = rng.choice(groups, size=n)

base = rng.normal(60, 12, size=n)
city_shift = {"Göteborg": 2, "Malmö": -1, "Stockholm": 1, "Uppsala": 0}
gender_shift = {"F": 1.5, "M": 0}
score = base + np.vectorize(city_shift.get)(city) + np.vectorize(gender_shift.get)(gender)
score = score + np.where(group == "B", 2.0, 0.0)
score = np.clip(score, 10, 100)

p_buy = 1 / (1 + np.exp(-(score - 55) / 10))
purchase = rng.binomial(1, p_buy)
revenue = np.where(purchase == 1, rng.lognormal(mean=3.2, sigma=0.5, size=n), 0.0)

df = pd.DataFrame({
    "city": city,
    "gender": gender,
    "age": age,
    "group": group,
    "score": np.round(score, 1),
    "purchase": purchase,
    "revenue": np.round(revenue, 2)
})

mask = rng.random(size=df.shape) < 0.01
for (row, column), is_missing in np.ndenumerate(mask):
    if is_missing:
        df.iat[row, column] = np.nan

df.to_csv("customer_activity.csv", index=False)
print("Skapade customer_activity.csv med", len(df), "rader.")

### Om datasetet (`customer_activity.csv`)

Det här är samma syntetiska dataset från den fiktiva digitala tjänsten som användes under vecka 5 och 6. Grupp B har en liten positiv effekt på `score`, och sannolikheten för ett köp ökar med högre `score`.

| Kolumn | Typ | Beskrivning |
|---|---|---|
| `city` | kategori | Kundens stad |
| `gender` | kategori | Kön (F/M) |
| `age` | heltal | Ålder |
| `group` | kategori | Grupp A eller B |
| `score` | decimaltal | Aktivitet eller engagemang, 10–100 |
| `purchase` | 0/1 | Om kunden genomförde ett köp |
| `revenue` | decimaltal | Intäkt, 0 om inget köp genomfördes |

Eftersom datasetet är syntetiskt ska det användas för att träna analysmetoder. Resultaten beskriver inte verkliga kunder.

### Snabb kontroll

- Läs in `customer_activity.csv` i en DataFrame som heter `df`.
- Kontrollera datatyper och antal saknade värden.
- Kontrollera att `group` bara innehåller A och B samt att `purchase` bara innehåller 0 och 1, bortsett från saknade värden.

## Block A – A/B-test för andelar

### Uppgift 1 – Test av andelar

Undersök om andelen kunder som genomfört ett köp skiljer sig mellan grupp A och grupp B.

- Ta bort rader där `group` eller `purchase` saknas.
- Kontrollera att `purchase` bara innehåller 0 och 1.
- Beräkna antal observationer, antal köp och köpandel i respektive grupp.
- Formulera en nollhypotes och en tvåsidig mothypotes.
- Kontrollera att det förväntade antalet köp och icke-köp är tillräckligt stort i båda grupperna.
- Genomför ett tvåsidigt z-test för två oberoende andelar.
- Rapportera z-statistika, p-värde och slutsats vid signifikansnivån 5 procent.

**Tips:** `statsmodels.stats.proportion.proportions_ztest()` kan användas.

### Uppgift 2 – Skillnad, konfidensintervall och praktisk betydelse

Fortsätt med köpandelarna från föregående uppgift.

- Beräkna den absoluta skillnaden `p_B - p_A`.
- Uttryck skillnaden i procentenheter.
- Beräkna ett 95-procentigt konfidensintervall för skillnaden mellan andelarna.
- Beräkna relativ förändring för grupp B jämfört med grupp A, om andelen i A är större än 0.
- Skapa en figur som visar den observerade skillnaden och dess konfidensintervall tillsammans med en referenslinje vid 0.

Skriv en kort slutsats där du skiljer mellan statistisk signifikans och praktisk betydelse. Förklara också varför en relativ förändring kan låta stor även när den absoluta skillnaden är liten.

## Block B – Power och stickprovsstorlek

### Uppgift 3 – Planera ett A/B-test

Anta att köpandelen i grupp A normalt är 25 procent. Du vill planera ett tvåsidigt A/B-test med signifikansnivån 5 procent och 80 procents power.

- Beräkna hur många observationer som behövs per grupp för att kunna upptäcka en ökning från 25 till 28 procent.
- Upprepa beräkningen för en ökning från 25 till 27 procent.
- Upprepa beräkningen för en ökning från 25 till 30 procent.
- Samla resultaten i en tabell och skapa en figur som visar sambandet mellan effektstorlek och nödvändig stickprovsstorlek.
- Förklara varför mindre effekter kräver större stickprov.

**Tips:** Använd `proportion_effectsize()` och `NormalIndPower().solve_power()` från `statsmodels`. Avrunda alltid den beräknade stickprovsstorleken uppåt.

## Block C – Flerfaldig testning

### Uppgift 4 – Många tester och korrigerade p-värden

Anta att följande p-värden kommer från tio analyser som genomförts samtidigt:

```python
p_values = np.array([0.001, 0.009, 0.018, 0.031, 0.047, 0.072, 0.11, 0.24, 0.51, 0.88])
```

- Avgör vilka tester som är signifikanta utan korrigering vid `alpha = 0.05`.
- Korrigera p-värdena med Bonferronimetoden.
- Korrigera dem även med Benjamini–Hochberg-metoden för FDR.
- Skapa en DataFrame med ursprungligt p-värde, korrigerat p-värde och beslut för båda metoderna.
- Jämför hur många hypoteser som förkastas med respektive metod.
- Förklara varför risken för minst ett falskt positivt resultat ökar när många tester genomförs.

**Tips:** Funktionen `statsmodels.stats.multitest.multipletests()` kan användas med metoderna `"bonferroni"` och `"fdr_bh"`.